In [1]:
import kagglehub
import pandas as pd
import numpy as np

path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("=" * 120)  
print(f"Directorio del archivo: {path}")
print("=" * 120)  

c:\Users\Pablito\Desktop\2026.1_Aprendizaje_Supervisado\control_2_ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Directorio del archivo: C:\Users\Pablito\.cache\kagglehub\datasets\camnugent\california-housing-prices\versions\1


In [2]:
df = pd.read_csv(path + "/housing.csv")
print("=" * 80)
print(df.head())
print("=" * 80)      

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


## Ítem a)

Introducimos $t_i \geq 0$ para cada observación $i = 1,...,N$ donde:

$$t_i = \left| y_i - w_0 - \sum_{j=1}^{8} w_j x_j^{(i)} \right|$$
Formulación LP:

$$\min_{w_0,...,w_8,\ t_1,...,t_N} \sum_{i=1}^{N} t_i$$

Sujeto a:

$$
y_i - \left(w_0 + \sum_{j=1}^{8} w_j x_j^{(i)}\right) \le t_i 
\quad \forall i = 1,\dots,N
$$

$$
-\Big(y_i - \left(w_0 + \sum_{j=1}^{8} w_j x_j^{(i)}\right)\Big) \le t_i 
\quad \forall i = 1,\dots,N
$$



$$t_i \geq 0 \quad \forall i = 1,...,N \qquad w_j \in \mathbb{R} \quad \forall j = 0,...,8$$

## Ítem b) 

Dividimos el conjunto de datos en:
- Entrenamiento: 80% de las observaciones 
- Prueba: 20% restante

Además el código al ejecutarse en la siguiente celda (Modelo AMPL) se encontraron valores nulos, dado esto, se rellenan estos datos con el valor medio, de modo de representar los valores de los coeficientes

In [3]:
np.random.seed(8008)

df_b = pd.get_dummies(df, columns=["ocean_proximity"], drop_first=True)

df_b = df_b.fillna(df_b.median())

indice = np.random.permutation(len(df_b))
n_entrenamiento = int(0.8*len(df_b))

entreno = df_b.iloc[indice[:n_entrenamiento]]
testeo  = df_b.iloc[indice[n_entrenamiento:]]

x_e = entreno.drop(columns=["median_house_value"])
x_t = testeo.drop(columns=["median_house_value"])

y_e = entreno["median_house_value"]
y_t = testeo["median_house_value"]

print("=" * 30)
print(f"Total observaciones : {len(df_b)}")
print(f"Observaciones train : {len(x_e)}")
print(f"Observaciones test  : {len(x_t)}")
print(f"Numero de features  : {x_e.shape[1]}")
print("=" * 30)

Total observaciones : 20640
Observaciones train : 16512
Observaciones test  : 4128
Numero de features  : 12


## Ítem c) 

Considerando los datos previamente procesados, se realiza la creación del modelo ADR con el lenjuage AMPL 

In [4]:
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=["highs", "gurobi", "xpress", "cplex"],
    license_uuid="ab452011-fa97-4a26-b111-d315c0cb941b"
)

np_x_e = x_e.values
np_y_e = y_e.values
np_x_e = np_x_e.astype(float)
np_y_e = np_y_e.astype(float)

I, J = np_x_e.shape

ampl.eval("""
    set I;
    set J;

    param x {i in I, j in J};
    param y {i in I};

    var w {j in J};
    var w0;
    var t {i in I} >= 0;

    minimize ADR:
        sum {i in I} t[i];

    subject to cota_superior {i in I}:
        y[i] - w0 - sum {j in J} w[j] * x[i,j] <= t[i];

    subject to cota_inferior {i in I}:
        -(y[i] - w0 - sum {j in J} w[j] * x[i,j]) <= t[i];
""")

ampl.set["I"] = range(1, I+1)
ampl.set["J"] = range(1, J+1)

x_ampl = {}
for i in range(I):
    for j in range(J):
        x_ampl[(i+1, j+1)] = np_x_e[i, j]

ampl.param["x"].set_values(x_ampl)

y_ampl = {}

for i in range(I):
    y_ampl[i+1] = np_y_e[i]
    
ampl.param["y"].set_values(y_ampl)

print("=" * 80)
ampl.solve(solver="gurobi")
print("Estado:", ampl.solve_result)
print("=" * 80)

Licensed to AMPL Community Edition License for <pablo.mellado@mail.udp.cl>.
Gurobi 13.0.1:Gurobi 13.0.1: optimal solution; objective 800986976.8
21 simplex iterations
18 barrier iterations
Estado: solved


Se extraen los coeficientes óptimos obtenidos desde AMPL para el modelo ADR.

In [5]:
w_vals = ampl.get_variable("w").get_values().to_pandas()
w0_val = ampl.get_value("w0")

coeficientes = []
variables = []
valores = []

coeficientes.append("w0")
variables.append("sesgo")
valores.append(w0_val)

for j in range(J):
    coeficientes.append("w" + str(j+1))
    variables.append(x_e.columns[j])
    valores.append(w_vals.iloc[j, 0])

resultados = pd.DataFrame({"Coeficiente": coeficientes, "Variable": variables, "Valor": valores})
print("=" * 60)
print("               COEFICIENTES DEL MODELO ADR")
print("=" * 60)
print(resultados)
print("=" * 60)

               COEFICIENTES DEL MODELO ADR
   Coeficiente                    Variable         Valor
0           w0                       sesgo -1.631745e+06
1           w1                   longitude -1.864301e+04
2           w2                    latitude -1.616832e+04
3           w3          housing_median_age  8.025097e+02
4           w4                 total_rooms -7.729807e+00
5           w5              total_bedrooms  8.137358e+01
6           w6                  population -3.696390e+01
7           w7                  households  7.789966e+01
8           w8               median_income  4.004007e+04
9           w9      ocean_proximity_INLAND -4.404356e+04
10         w10      ocean_proximity_ISLAND  1.136255e+05
11         w11    ocean_proximity_NEAR BAY -7.543683e+03
12         w12  ocean_proximity_NEAR OCEAN  9.162717e+03


## Ítem d) 

La solución óptima se obtiene directamente mediante la **ecuación normal**:

$$w^* = (X^\top X)^{-1} X^\top y$$

Donde $X$ es la matriz de diseño con una columna de unos agregada para el intercepto $w_0$.

In [ ]:

list_uno  = np.ones((len(x_e), 1))
d_x = np.hstack([list_uno , np_x_e])
print(d_x)
XtX   = d_x.T @ d_x          
XtX_inv = np.linalg.inv(XtX)     
Xty   = d_x.T @ np_y_e        

d_w = XtX_inv @ Xty
print(d_w)

[[   1.   -116.95   33.75 ...    0.      0.      0.  ]
 [   1.   -121.45   38.56 ...    0.      0.      0.  ]
 [   1.   -118.28   33.73 ...    0.      0.      1.  ]
 ...
 [   1.   -122.23   37.76 ...    0.      1.      0.  ]
 [   1.   -118.33   33.88 ...    0.      0.      0.  ]
 [   1.   -117.88   33.72 ...    0.      0.      0.  ]]
[-2.16266434e+06 -2.54574370e+04 -2.40345691e+04  1.12475959e+03
 -5.21077640e+00  7.57404510e+01 -3.97117527e+01  7.89149711e+01
  3.93368407e+04 -3.99600570e+04  1.31712741e+05 -4.16400828e+03
  5.45198379e+03]


In [ ]:
from sklearn.linear_model import LinearRegression

modelo_sk = LinearRegression()
modelo_sk.fit(np_x_e, np_y_e)

coeficientes = []
variables    = []
valores_OLS  = []
valores_adr  = []
valores_sk   = []

coeficientes.append("w0")
variables.append("sesgo")
valores_OLS.append(d_w[0])
valores_adr.append(w0_val)
valores_sk.append(modelo_sk.intercept_)

for j in range(J):
    coeficientes.append("w" + str(j+1))
    variables.append(x_e.columns[j])
    valores_OLS.append(d_w[j+1])
    valores_adr.append(w_vals.iloc[j, 0])
    valores_sk.append(modelo_sk.coef_[j])

comparacion = pd.DataFrame({
    "Coeficiente" : coeficientes,
    "Variable"    : variables,
    "OLS"         : valores_OLS,
    "ADR"         : valores_adr,
    "OLS sklearn" : valores_sk
})

print("=" * 90)
print("      COMPARACION DE COEFICIENTES: OLS vs ADR vs OLS sklearn")
print("=" * 90)
print(comparacion)
print("=" * 90)


      COMPARACION DE COEFICIENTES: OLS vs ADR vs OLS sklearn
   Coeficiente                    Variable           OLS           ADR  \
0           w0                       sesgo -2.162664e+06 -1.631745e+06   
1           w1                   longitude -2.545744e+04 -1.864301e+04   
2           w2                    latitude -2.403457e+04 -1.616832e+04   
3           w3          housing_median_age  1.124760e+03  8.025097e+02   
4           w4                 total_rooms -5.210776e+00 -7.729807e+00   
5           w5              total_bedrooms  7.574045e+01  8.137358e+01   
6           w6                  population -3.971175e+01 -3.696390e+01   
7           w7                  households  7.891497e+01  7.789966e+01   
8           w8               median_income  3.933684e+04  4.004007e+04   
9           w9      ocean_proximity_INLAND -3.996006e+04 -4.404356e+04   
10         w10      ocean_proximity_ISLAND  1.317127e+05  1.136255e+05   
11         w11    ocean_proximity_NEAR BAY -4.16400

## Ítem e)

El modelo LSR4 minimiza la suma de residuos a la cuarta potencia:

$$\min_{w} \sum_{i=1}^{N} \left( y_i - f(x^{(i)}) \right)^4$$

$$\min_{w} \sum_{i=1}^{N} \left( y_i - w_0 - \sum_{j=1}^{n} w_j x_j^{(i)} \right)^4$$

Usamos descenso de gradiente que en cada 
iteración actualiza los pesos moviéndose en la dirección que reduce el error:

$$w^{k+1} = w^k - \alpha \nabla f(w^k)$$

Para aplicar esto necesitamos el gradiente, que se calcula derivando la función 
objetivo respecto a cada peso:

$$\nabla_w J = -4 \sum_{i=1}^{N} \left( y_i - f(x^{(i)}) \right)^3 x^{(i)}$$

Reemplazando el gradiente en la regla de actualización, cada iteración queda:

$$w^{k+1} = w^k + 4\alpha \sum_{i=1}^{N} \left( y_i - f^k(x^{(i)}) \right)^3 x^{(i)}$$



In [ ]:
e_x = d_x

alpha = 1e-12000